# Experiment 1

In [0]:
from pyspark.sql.functions import col, input_file_name, regexp_extract, sum, count, stddev, when, row_number, min
from pyspark.sql.window import Window

RAW_CSV_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/set/depression/"
FORMS_PATH = "/Volumes/workspace/anima/anima_volume/raw/dynamic_ab_research/forms.csv"

BRONZE_TABLE = "workspace.anima.bronze_eye_tracking"
SILVER_TABLE = "workspace.anima.silver_eye_tracking"
GOLD_TABLE   = "workspace.anima.gold_features"

## Creating BRONZE_TABLE

In [0]:
from pyspark.sql.functions import col

print(f"Reading raw CSVs from {RAW_CSV_PATH}...")

bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(RAW_CSV_PATH)
    .select("*", col("_metadata.file_path").alias("source_file"))
)

print("Writing to Bronze Table...")
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(BRONZE_TABLE)

display(spark.table(BRONZE_TABLE).limit(5))

## Creating SILVER_TABLE

In [0]:
bronze = spark.table(BRONZE_TABLE)

silver_df = bronze.withColumn("sid", regexp_extract("source_file", r"\/([^\/]+)\.csv$", 1))

forms = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(FORMS_PATH)
    .select("sid", "uid", "phq-9_score", "createdAt")
)

window_spec = Window.partitionBy("uid").orderBy("createdAt")

forms_filtered = (
    forms.withColumn("session_rank", row_number().over(window_spec))
    .filter(col("session_rank") == 1)
    .drop("session_rank", "createdAt")
)

silver_joined = silver_df.join(forms_filtered, on="sid", how="inner")

print("Writing to Silver Table...")
silver_joined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

print(f"Silver Table Created. Rows: {silver_joined.count()}")

## Creating GOLD_TABLE

# Proposed Solution: Static Attention Indicators

This section defines the static attention indicators used to construct the feature vectors for the Machine Learning models. The implementation below corresponds to the mathematical definitions provided in the thesis.

## 1. Fixation Measures
* **Fixation Count** (`fixation_count`)
    * *Definition:* The total number of fixation events identified within the session.
    * *Code:* Sum of `is_fixation_end` events.
* **Fixation Duration** (`avg_fixation_duration`)
    * *Definition:* The average time (ms) the eyes remained stationary per fixation.
    * *Code:* Average of `FDUR` (Fixation Duration) for valid fixations.
* **Fixation Rate** (`fixation_rate`)
    * *Definition:* The frequency of fixations per unit of time (fixations per second).
    * *Code:* `fixation_count` divided by session duration in seconds.

## 2. Saccade Measures
* **Saccade Amplitude** (`avg_saccade_amplitude`)
    * *Definition:* The average distance traveled during eye movements (scanpath).
    * *Code:* `total_scanpath_dist` divided by `fixation_count`.
* **Saccade Velocity** (`avg_saccade_velocity`)
    * *Definition:* The average speed of eye movement (pixels/ms).
    * *Code:* `total_scanpath_dist` divided by `movement_time_ms` (Time spent moving).
* **Saccade Rate** (`saccade_rate`)
    * *Definition:* The number of saccadic movements per unit time.
    * *Code:* Implemented using `fixation_rate` as a direct proxy (1:1 relationship).

## 3. Blink Measures
* **Blink Rate** (`blink_rate`)
    * *Definition:* The number of blinks per minute.
    * *Code:* `blink_count` divided by session duration in minutes.
* **Average Blink Duration** (`avg_blink_duration`)
    * *Definition:* The average time the eyes remained closed during a blink.
    * *Code:* `total_blink_ms` divided by `blink_count`.
* **PERCLOS / Alertness** (`blink_time_perc`)
    * *Definition:* The percentage of total session time spent with eyes closed.
    * *Code:* `total_blink_ms` divided by `total_ms`.

## 4. Visual Search Measures
* **Time-to-First-Fixation** (`avg_time_to_first_fixation_ms`)
    * *Definition:* The average latency between the onset of a new scene and the first fixation on that scene.
    * *Code:* Calculated via Multi-Scene aggregation (Latency averaged across all scenes).
* **Normalized Dwell Time** (`norm_dwell_time_perc`)
    * *Definition:* The percentage of the session spent actively processing visual information (fixating).
    * *Code:* `total_fixation_ms` divided by `total_ms`.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, sum, col, when, stddev, avg, sqrt, pow, min, max, first

silver = spark.table(SILVER_TABLE)
window_spec = Window.partitionBy("sid").orderBy("TIMESTAMP")

w_scene = Window.partitionBy("sid", "SCENE_INDEX")
ttff_prep = (
    silver
    .withColumn("scene_start_time", min("TIMESTAMP").over(w_scene))
    .filter(col("FEV") == 3)
    .withColumn("fix_start_time", col("TIMESTAMP") - col("FDUR"))
    .filter(col("fix_start_time") >= col("scene_start_time"))
)

scene_ttff = (
    ttff_prep
    .groupBy("sid", "SCENE_INDEX")
    .agg(
        min("fix_start_time").alias("first_fix_ts"),
        first("scene_start_time").alias("scene_start_ts")
    )
    .withColumn("scene_latency_ms", col("first_fix_ts") - col("scene_start_ts"))
)

user_avg_ttff = (
    scene_ttff.groupBy("sid")
    .agg(avg("scene_latency_ms").alias("avg_ttff_ms"))
)

silver_enhanced = silver.withColumn(
    "prev_blink", lag("BLINK", 1, False).over(window_spec)
).withColumn(
    "is_blink_start", when((col("BLINK") == True) & (col("prev_blink") == False), 1).otherwise(0)
).withColumn(
    "is_fixation_end", when(col("FEV") == 3, 1).otherwise(0)
).withColumn(
    "step_distance", 
    sqrt(pow(col("BPOGX") - lag("BPOGX", 1).over(window_spec), 2) + 
         pow(col("BPOGY") - lag("BPOGY", 1).over(window_spec), 2))
)

gold_agg = (
    silver_enhanced.groupBy("sid", "uid", "phq-9_score")
    .agg(
        (sum("ROW_DURATION") / 1000 / 60).alias("minutes"),
        (sum("ROW_DURATION")).alias("total_ms"),
        
        sum("is_blink_start").alias("blink_count"),
        sum(when(col("BLINK") == True, col("ROW_DURATION")).otherwise(0)).alias("total_blink_ms"),
        
        sum("is_fixation_end").alias("fixation_count"),
        sum(when(col("FEV") == 3, col("FDUR"))).alias("total_fixation_ms"),
        avg(when(col("FEV") == 3, col("FDUR"))).alias("avg_fixation_ms"),
        
        sum("step_distance").alias("total_scanpath_dist")
    )
)

gold_joined = gold_agg.join(user_avg_ttff, "sid", "left")

gold_features = (
    gold_joined
    .withColumn("blink_rate", when(col("minutes") > 0, col("blink_count") / col("minutes")).otherwise(0))
    .withColumn("avg_blink_duration", when(col("blink_count") > 0, col("total_blink_ms") / col("blink_count")).otherwise(0))
    .withColumn("blink_time_perc", when(col("total_ms") > 0, (col("total_blink_ms") / col("total_ms")) * 100).otherwise(0))
    
    .withColumn("fixation_rate", when(col("minutes") > 0, col("fixation_count") / (col("minutes") * 60)).otherwise(0)) 
    .withColumn("avg_fixation_duration", col("avg_fixation_ms"))
    .withColumn("norm_dwell_time_perc", when(col("total_ms") > 0, (col("total_fixation_ms") / col("total_ms")) * 100).otherwise(0))
    
    .withColumn("saccade_rate", col("fixation_rate"))
    .withColumn("avg_saccade_amplitude", 
        when(col("fixation_count") > 0, col("total_scanpath_dist") / col("fixation_count")).otherwise(0)
    )
    .withColumn("movement_time_ms", col("total_ms") - col("total_fixation_ms") - col("total_blink_ms"))
    .withColumn("avg_saccade_velocity", 
        when(col("movement_time_ms") > 0, col("total_scanpath_dist") / col("movement_time_ms")).otherwise(0)
    )

    .withColumn("avg_time_to_first_fixation_ms", when(col("avg_ttff_ms").isNotNull(), col("avg_ttff_ms")).otherwise(0))

    .withColumn("severity_group", 
        when(col("phq-9_score") <= 5, "1. Minimal (0-5)")
        .when((col("phq-9_score") > 5) & (col("phq-9_score") <= 10), "2. Mild (6-10)")
        .when((col("phq-9_score") > 10) & (col("phq-9_score") <= 15), "3. Moderate (11-15)")
        .otherwise("4. Severe (16+)")
    )
)

gold_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)

print("✅ Gold Table Updated with Multi-Scene TTFF.")
display(gold_features)

## Property visualization

In [0]:
from visualization import visualize_severity_trend

df = spark.table(GOLD_TABLE)
print("Data Loaded.")

### Blink rate

In [0]:
visualize_severity_trend(df, "blink_rate", title="Blink Rate (Blinks per Minute)")

### Average Blink Duration

In [0]:
visualize_severity_trend(df, "avg_blink_duration", title="Average Blink Duration (ms)")

### Blink Time %

In [0]:
visualize_severity_trend(df, "blink_time_perc", title="Percentage of Time with Eyes Closed (PERCLOS)")

### Fixation Rate

In [0]:
visualize_severity_trend(df, "fixation_rate", title="Fixation Rate (Events per Second)")

### Average Fixation Duration

In [0]:
visualize_severity_trend(df, "avg_fixation_duration", title="Average Fixation Duration (ms)")

### Normalized Dwell Time %

In [0]:
visualize_severity_trend(df, "norm_dwell_time_perc", title="Normalized Visual Processing Time (%)")

### Saccade Rate

In [0]:
visualize_severity_trend(df, "saccade_rate", title="Saccade Rate (Moves per Second)")

### Average Saccade Amplitude

In [0]:
visualize_severity_trend(df, "avg_saccade_amplitude", title="Average Saccade Amplitude (Pixels)")

### Average Saccade Velocity

In [0]:
visualize_severity_trend(df, "avg_saccade_velocity", title="Average Saccade Velocity (Pixels/ms)")

### Average Time to First Fixation

In [0]:
visualize_severity_trend(df, "avg_time_to_first_fixation_ms", title="Avg Time to First Fixation (Reaction Latency ms)")

## Linear Regression (PHQ-9 Score Prediction)

####### Experiment 1: Linear Regression (PHQ-9 Score Prediction)

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

df = spark.table(GOLD_TABLE).filter(col("minutes") > 1.0)
print(f"Data count after filtering duration: {df.count()}")

feature_cols = [
    "blink_rate", 
    "avg_blink_duration", 
    "blink_time_perc",
    
    "fixation_rate", 
    "avg_fixation_duration", 
    "norm_dwell_time_perc",
    
    "saccade_rate", 
    "avg_saccade_amplitude", 
    "avg_saccade_velocity", 
    
    "avg_time_to_first_fixation_ms"
]

df_clean = df.na.drop(subset=feature_cols)
print(f"Data count after dropping nulls: {df_clean.count()}")

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

lr = LinearRegression(featuresCol="features", labelCol="phq-9_score", regParam=0.1)

train_data, test_data = df_clean.randomSplit([0.8, 0.2], seed=42)

pipeline = Pipeline(stages=[assembler, scaler, lr])
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

evaluator_rmse = RegressionEvaluator(labelCol="phq-9_score", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="phq-9_score", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("-" * 50)
print(f"MODEL RESULTS (Linear Regression)")
print(f"RMSE (Error): {rmse:.2f} points (Lower is better)")
print(f"R2 (Explained Variance): {r2:.4f} (Higher is better, >0.1 is a start)")
print("-" * 50)

lr_model = model.stages[-1]
print("Feature Importance (Standardized Weights):")
print("(Positive = Higher Feature -> Higher Depression)")
print("(Negative = Higher Feature -> Lower Depression)")
print("-" * 50)

for feature, weight in zip(feature_cols, lr_model.coefficients):
    print(f"  {feature:<30}: {weight:.4f}")

if r2 < 0.1:
    print("\n⚠️ CONCLUSION: The model is struggling to predict the EXACT score.")
    print("   This is normal for physiological data. Next step: Try Classification (Groups).")
else:
    print("\n✅ CONCLUSION: The model found a signal!")

## Classification (Logistic Regression & Random Forest)

####### Experiment 2: Binary Classification (Logistic Regression vs. Random Forest)

In [0]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline

df = spark.table(GOLD_TABLE).filter(col("minutes") > 1.0)

feature_cols = [
    "blink_rate", "avg_blink_duration", "blink_time_perc",
    "fixation_rate", "avg_fixation_duration", "norm_dwell_time_perc",
    "saccade_rate", "avg_saccade_amplitude", "avg_saccade_velocity", 
    "avg_time_to_first_fixation_ms"
]

data = df.na.drop(subset=feature_cols)

data = data.withColumn("label", when(col("phq-9_score") >= 10, 1.0).otherwise(0.0))

print("Class Balance:")
data.groupBy("label").count().show()

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

lr = LogisticRegression(featuresCol="features", labelCol="label", regParam=0.1)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, seed=42)

train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("\n--- RUNNING LOGISTIC REGRESSION ---")
pipeline_lr = Pipeline(stages=[assembler, scaler, lr])
model_lr = pipeline_lr.fit(train_data)
pred_lr = model_lr.transform(test_data)

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc_lr = evaluator_auc.evaluate(pred_lr)
print(f"Logistic Regression AUC: {auc_lr:.3f}")

print("\n--- RUNNING RANDOM FOREST ---")
pipeline_rf = Pipeline(stages=[assembler, scaler, rf]) 
model_rf = pipeline_rf.fit(train_data)
pred_rf = model_rf.transform(test_data)

auc_rf = evaluator_auc.evaluate(pred_rf)
acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
acc_rf = acc_eval.evaluate(pred_rf)

print(f"Random Forest AUC: {auc_rf:.3f}")
print(f"Random Forest Accuracy: {acc_rf:.1%}")

print("-" * 50)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("(Higher number = The most critical biomarkers)")
print("-" * 50)

rf_model = model_rf.stages[-1]
importances = rf_model.featureImportances

feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

for feature, importance in feat_imp:
    print(f"  {feature:<30}: {importance:.4f}")

print("-" * 50)

if auc_rf > 0.55:
    print("✅ CONCLUSION: Random Forest is working better than Linear models.")
    print("   This proves the relationship is NON-LINEAR.")
else:
    print("⚠️ CONCLUSION: Both models are struggling. The signal might be highly individual.")